# Laptop lab 5: Catching fraud
*When accuracy lies, and the threshold is a choice*

**Machine Learning with Python** · Minor Amsterdam Data Science and Artificial Intelligence · Fall 2026<br>
Thursday, November 26, 2026 · Laptop labs: Batuhan Ataş (b.atas@uva.nl)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/batuatas/MLwPython-labs/blob/main/lab05.ipynb)

> **Before you start:** choose **File → Save a copy in Drive** and work in that copy. Colab does not save your changes in the original notebook.

This week you learned to measure a classifier properly (Lecture 9) and what to do when the class you look for is rare (Lecture 10). Today you use both on real card payments.

**The story.** A bank wants to stop card fraud. A model flags payments, and an analyst checks every flagged payment and calls the cardholder: a few minutes of work, so a false alarm is cheap. But every fraud the model misses is money stolen from a customer. Only about **1 payment in 100** is fraud. As always, some payments stay locked away: the **test set**, 20,000 payments.

**Your mission.** Build one pipeline, and choose its threshold, so that it catches **at least 85% of the frauds** (recall ≥ 0.85) with **as few false alarms as possible** (precision as high as possible).

## How this lab works

| You see | What you do |
|---|---|
| **Run** | Run the cell and look at the output. |
| **Your turn** | Write the code yourself, or fill in the dots `...`. A check below the cell shows an orange *Not yet* box until you are done. |
| **Toolbox** | Click to unfold a piece of code that you can copy and change. |
| **Think** | Double-click *Your answer* and write a sentence or two. We discuss the answers together. |
| **Stop** | Wait here until the tutor says you can go on. |

Today a model does not only say *fraud or not*; it says *how likely*. You choose where to draw the line:

```python
proba = model.predict_proba(X)[:, 1]   # the probability of fraud, for every payment
flag = proba >= 0.3                    # your threshold: flag a payment from 30% on
recall_score(y, flag)                  # the share of frauds that are flagged
precision_score(y, flag)               # the share of flagged payments that are fraud
```

| Part | Minutes |
|---|---|
| 0. Setup | 5 |
| 1–3. Warm-up: accuracy, the threshold, class weights | together on the screen, 15 |
| 4. Your mission | 45 |
| 5. The test set is unlocked | 5 |
| 6. What happened? | together, 15 |
| Check your understanding | together, at the end |

## Task 0. Setup

Run the cell below. It loads the libraries, and downloads the tools and the data of this lab. You should see a green box with a ✓.

In [ ]:
try:
    import imblearn                         # imbalanced-learn, for Lecture 10
except ImportError:
    !pip install -q imbalanced-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (recall_score, precision_score, average_precision_score,
                             precision_recall_curve, ConfusionMatrixDisplay)
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.ensemble import BalancedRandomForestClassifier

# The tools and the data of this lab are in the course lab repository.
BASE = "https://raw.githubusercontent.com/batuatas/MLwPython-labs/main/"
import urllib.request
urllib.request.urlretrieve(BASE + "lab05_tools.py", "lab05_tools.py")
from lab05_tools import *

X, y = load_training_data()    # your 30,000 payments: features X, and y = 1 for fraud
setup_ok()

**Run.** Your data: card payments of European cardholders over two days in September 2013. `Time` counts the seconds since the first payment, and `Amount` is the amount paid. To protect the cardholders, the bank did not publish the original features (shop, country, card type, ...), only 28 anonymous mixtures of them: `V1` to `V28`. There are no gaps and no text columns, so you can start right away.

In [ ]:
print(X.shape, "  frauds:", y.sum(), f"({y.mean():.1%})")
X.head()

## Warm-up: the pieces, one by one (Tasks 1 to 3)

We go through Tasks 1 to 3 together on the screen: just run the cells and follow along. You will reuse them in your mission.

## Task 1. Accuracy lies

**Run.** A model that never flags anything, against a logistic regression. Which one is better? (Lecture 9, "Is 0.87 good?")

In [ ]:
never = DummyClassifier()                                               # always "not fraud"
logreg = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

for name, model in [("never fraud", never), ("logistic regression", logreg)]:
    print(f"{name:20} mean validation accuracy {cross_val_score(model, X, y, cv=5).mean():.3f}")

Doing nothing is 99% accurate: 99 in 100 payments are honest anyway. The logistic regression looks only a little better. Accuracy hides what the bank cares about. Let us count the mistakes instead.

**Run.** `cross_val_predict` gives, for every payment, the prediction of the model that did *not* see it during training: an honest prediction for all 30,000 payments at once. The confusion matrix counts the four outcomes (Lecture 9, "The confusion matrix").

In [ ]:
predicted = cross_val_predict(logreg, X, y, cv=5)
ConfusionMatrixDisplay.from_predictions(y, predicted, display_labels=["honest", "fraud"])
plt.show()
print(f"recall    {recall_score(y, predicted):.2f}   (the share of frauds that are caught)")
print(f"precision {precision_score(y, predicted):.2f}   (the share of flagged payments that are fraud)")

When the model flags a payment, it is almost always right. But it lets about one fraud in five through: below the goal of 85%.

## Task 2. The threshold is a choice

**Run.** `predict` flags a payment when the probability of fraud is at least 0.5. That is a choice, not a law (Lecture 9, "The threshold is a choice"). Lower the threshold, and watch recall and precision:

In [ ]:
proba = cross_val_predict(logreg, X, y, cv=5, method="predict_proba")[:, 1]   # honest probabilities

for threshold in [0.5, 0.1, 0.05, 0.02, 0.01]:
    flag = proba >= threshold
    print(f"threshold {threshold:.2f}: flags {flag.sum():5} payments,  "
          f"recall {recall_score(y, flag):.2f},  precision {precision_score(y, flag):.2f}")

A lower threshold catches more frauds, and raises more false alarms: slowly at first, then fast. **Run.** The precision-recall curve shows every threshold at once; the dot is the threshold 0.5. Average precision summarizes the whole curve in one number: 1 is perfect, and a model that guesses gets the share of frauds, 0.01 (Lecture 9, "The precision-recall curve").

In [ ]:
precision, recall, thresholds = precision_recall_curve(y, proba)
flag = proba >= 0.5

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.plot(recall, precision, label=f"logistic regression (average precision {average_precision_score(y, proba):.2f})")
ax.plot(recall_score(y, flag), precision_score(y, flag), "o", color="black", label="threshold 0.5")
ax.axvline(0.85, color="grey", linestyle="--", label="the goal: recall 0.85")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.legend(loc="lower left")
plt.show()

To reach the goal, walk along the curve to the right until recall is 0.85, and read off the precision there. Just after the goal, the curve falls off a cliff: the last frauds look like honest payments. That is Step 2 of your mission.

## Task 3. Class weights move the dot, not the curve

**Run.** The cheapest remedy of Lecture 10: `class_weight="balanced"` makes every fraud count as much as about a hundred honest payments during training (Lecture 10, "Remedy 2: class weights"). Compare the two models, at the threshold 0.5 and as curves:

In [ ]:
weighted = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced"))
proba_w = cross_val_predict(weighted, X, y, cv=5, method="predict_proba")[:, 1]

fig, ax = plt.subplots(figsize=(5.5, 4.5))
for name, p, color in [("plain", proba, "#C0002A"), ("class weights", proba_w, "#25567B")]:
    pr, rc, _ = precision_recall_curve(y, p)
    flag = p >= 0.5
    ax.plot(rc, pr, color=color, label=f"{name} (average precision {average_precision_score(y, p):.2f})")
    ax.plot(recall_score(y, flag), precision_score(y, flag), "o", color=color, markersize=9)
    print(f"{name:14} at 0.5: recall {recall_score(y, flag):.2f}, precision {precision_score(y, flag):.2f}")
ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.legend(loc="lower left")
plt.show()

At 0.5, the weighted model catches nine frauds in ten, but only about one flagged payment in four is fraud. The two curves lie almost on top of each other: the weights did not make the model better, they moved its dot along the same curve, just as a lower threshold would (Lecture 10, "What did we buy?"). Remember this in your mission.

## Task 4. Your mission

Now it is your turn. Build **one pipeline** and choose **one threshold** so that, in cross-validation, it catches **at least 85% of the frauds** with the **highest precision** you can get.

You do this in four short steps. Step 1 is ready to run. In Steps 2 to 4 the comments in each cell say what to do; copy from Tasks 1 to 3, from Step 1 and from the **toolbox** right below. After every step, write down two numbers: the threshold that reaches recall 0.85, and the precision there.

**The rule:** you may do anything with `X` and `y`. The test set stays locked.

<details><summary><b>Toolbox</b> (click to unfold)</summary>

### 1. Honest probabilities, and the threshold for the goal

`cross_val_predict` gives every payment a probability from a model that did not see it. On these, find the highest threshold that still reaches recall 0.85: the highest, because every lower threshold only adds false alarms.

```python
proba = cross_val_predict(my_pipe, X, y, cv=5, method="predict_proba")[:, 1]

precision, recall, thresholds = precision_recall_curve(y, proba)
reach = np.where(recall[:-1] >= 0.85)[0][-1]       # the last threshold with recall >= 0.85
print(f"threshold {thresholds[reach]:.3f}: recall {recall[reach]:.2f}, precision {precision[reach]:.2f}")
```

### 2. Class weights

One argument of the model; frauds then count more during training.

```python
LogisticRegression(max_iter=1000, class_weight="balanced")
RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=0, n_jobs=-1)
```

### 3. Resampling, inside the pipeline

Change the training data instead of the model (Lecture 10, "Three ways to resample"). Use the pipeline of **imbalanced-learn**, `make_imb_pipeline`: it resamples the training folds only, never the payments the model is tested on (Lecture 10, "Resample the training data only").

```python
make_imb_pipeline(StandardScaler(), SMOTE(random_state=0), LogisticRegression(max_iter=1000))               # new frauds, made up between neighbors
make_imb_pipeline(StandardScaler(), RandomUnderSampler(random_state=0), LogisticRegression(max_iter=1000))  # drop honest payments until 50/50
```

SMOTE measures distances, so the scaler comes first.

### 4. Forests

A forest needs no scaler, so the model alone is enough. A balanced random forest gives every tree its own balanced sample (Lecture 10, "Undersampling inside a forest"). Each takes up to a minute in cross-validation.

```python
RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1)
BalancedRandomForestClassifier(n_estimators=100, sampling_strategy="all", replacement=True,
                               bootstrap=False, random_state=0, n_jobs=-1)
```

### 5. Comparing models without a threshold

Average precision scores the whole curve (Lecture 9, "Comparing models without choosing a threshold"). Compute it from the honest probabilities you already have:

```python
average_precision_score(y, proba)
```

</details>

### Step 1. The starting pipeline

**Run.** The logistic regression of Task 1, at the usual threshold 0.5. Does it reach the goal?

In [ ]:
# Step 1. Run this cell. Nothing to change.
my_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
proba = cross_val_predict(my_pipe, X, y, cv=5, method="predict_proba")[:, 1]
flag = proba >= 0.5
print(f"Step 1, threshold 0.5: recall {recall_score(y, flag):.2f}, precision {precision_score(y, flag):.2f}")

*Stuck?* Cheat sheet, "Precision, recall, F1" · Lecture 9, "Precision and recall".

### Step 2. Choose the threshold

Keep the model of Step 1, and lower its threshold until it reaches the goal. What precision is left?

In [ ]:
# Step 2. The threshold for recall 0.85 (Toolbox, part 1).
# 1. Use the probabilities `proba` of Step 1 and precision_recall_curve.
# 2. Find the highest threshold with recall >= 0.85.
# 3. Print that threshold, its recall and its precision.

*Stuck?* Toolbox, part 1 · Cheat sheet, "Probabilities and the threshold" · Lecture 10, "Remedy 1: lower the threshold".

### Step 3. A remedy of Lecture 10

Try class weights, SMOTE or undersampling. Look at two things: the recall and precision at 0.5, and the precision at recall 0.85 (Step 2 again). Does the remedy beat Step 2?

In [ ]:
# Step 3. A remedy (Toolbox, parts 2 and 3).
# 1. Make a pipeline with class weights, SMOTE or undersampling.
# 2. Get its honest probabilities with cross_val_predict, as in Step 1.
# 3. Print recall and precision at 0.5, then find the threshold for recall 0.85 as in Step 2.

*Stuck?* Toolbox, parts 2 and 3 · Cheat sheet, "A rare class" · Lecture 10, "Remedy 2: class weights" and "SMOTE: synthetic examples".

### Step 4. A forest

In Lecture 10, the step from logistic regression to a forest mattered more than any remedy. Does it here? Compare the average precision, and the precision at recall 0.85.

In [ ]:
# Step 4. A forest (Toolbox, parts 4 and 5).
# 1. Make a random forest, or a balanced random forest.
# 2. Get its honest probabilities with cross_val_predict, as in Step 1.
# 3. Compare its average precision with the logistic regression of Step 1.
# 4. Find the threshold for recall 0.85 and its precision, as in Step 2.

*Stuck?* Toolbox, parts 4 and 5 · Cheat sheet, "Curves" and "A rare class" · Lecture 10, "Undersampling inside a forest" and "The forests as curves".

### Your notes

**Your turn.** Double-click the cell below and write down, for each step, the threshold for recall 0.85 and the precision there. Then choose your pipeline and its threshold.

*Your notes:*

- Step 1, threshold 0.5: ...
- Step 2, threshold for recall 0.85 and its precision: ...
- Step 3, my remedy: ...
- Step 4, a forest: ...
- My choice: ...

### Your submission

**Your turn.** Fill in your pipeline, its threshold, and the recall and precision you expect on the locked payments (your cross-validated numbers are good estimates). The last line fits your pipeline once more on **all** 30,000 training payments.

In [ ]:
my_pipeline = ...      # your best pipeline, for example my_pipe (Step 1)
my_threshold = ...     # flag a payment as fraud from this probability on, for example 0.05
my_recall = ...        # the recall you expect on new payments: at least 0.85
my_precision = ...     # the precision you expect on new payments

my_pipeline.fit(X, y)  # learn from all 30,000 training payments

In [ ]:
# Check
check_submission()

> **Stop.** Wait here until the tutor unlocks the test set. Meanwhile, compare with your neighbor: the same model? The same threshold?

## Task 5. The test set is unlocked

**Your turn.** When the tutor shows the password on the screen, type it in quotes and run the cell. Then run the next cell: it flags the 20,000 payments that no model has seen with your threshold, and counts.

In [ ]:
X_test, y_test = unlock_test_set("...")    # the password on the screen

In [ ]:
test_recall, test_precision = report(my_pipeline, my_threshold, my_recall, my_precision, X_test, y_test)

Tell the tutor your pipeline, your threshold, and your recall and precision on the test set. We collect them on the screen: who met the goal, and with how many false alarms?

## Task 6. What happened?

**Run.** The pieces of today on the test set: each pipeline at the threshold 0.5, and its average precision. This takes about a minute.

In [ ]:
check_unlocked(X_test)      # stops here while the test set is still locked
candidates = {
    "never fraud": never,
    "logistic regression": logreg,
    "  + class weights": weighted,
    "  + SMOTE": make_imb_pipeline(StandardScaler(), SMOTE(random_state=0), LogisticRegression(max_iter=1000)),
    "random forest": RandomForestClassifier(100, random_state=0, n_jobs=-1),
    "balanced random forest": BalancedRandomForestClassifier(
        100, sampling_strategy="all", replacement=True, bootstrap=False, random_state=0, n_jobs=-1),
}
rows = []
for name, m in candidates.items():
    m.fit(X, y)
    p = m.predict_proba(X_test)[:, 1]
    rows.append({"pipeline": name,
                 "accuracy": ((p >= 0.5) == y_test).mean(),
                 "recall at 0.5": recall_score(y_test, p >= 0.5),
                 "precision at 0.5": precision_score(y_test, p >= 0.5, zero_division=0),
                 "average precision": average_precision_score(y_test, p)})
pd.DataFrame(rows).set_index("pipeline").round(3)

**Think.** We discuss these together.

(a) Which pipelines have an accuracy above 0.99, and would the bank want each of them? (Lecture 9, "Which metric?")<br>
(b) With class weights or SMOTE, recall at 0.5 goes up from about 0.84 to about 0.91, and precision falls from about 0.95 to about 0.25. Average precision hardly moves. What does that tell you? (Lecture 10, "The forests as curves")<br>
(c) You chose your threshold so that recall was 0.85 in cross-validation. Why not aim at 0.95, to be safe? Look at the curve of Task 2.<br>
(d) A shop wants to block a payment at the till, with no analyst in between, and a wrong block loses a customer. Which metric matters now, and would you move the threshold up or down?<br>
(e) Suppose a check costs the bank €5 and a missed fraud €100. What did your pipeline cost on the test set, and what would "never fraud" have cost?

*Your answer:*

## Check your understanding

Answer each question for yourself first, then unfold the answer. We go through them together at the end of the lab.

**1.** One patient in a hundred has a rare disease. A model says "healthy" for everybody. What are its accuracy and its recall?

A. 0.99 and 0.99 &nbsp; B. 0.99 and 0 &nbsp; C. 0.01 and 0 &nbsp; D. 0.5 and 0.5

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** It is right for the 99 healthy patients, so its accuracy is 0.99, but it finds none of the sick ones: recall 0 (Lecture 10, "First, evaluate correctly").

</details>

**2.** A fraud model flags 100 payments; 20 of them are fraud. The data contain 50 frauds. What are precision and recall?

A. 0.20 and 0.40 &nbsp; B. 0.40 and 0.20 &nbsp; C. 0.20 and 0.50 &nbsp; D. 0.80 and 0.40

<details><summary><b>Answer</b> (click to unfold)</summary>

**A.** Precision: 20 of the 100 flagged are fraud, 0.20. Recall: 20 of the 50 frauds are found, 0.40 (Lecture 9, "Precision and recall").

</details>

**3.** Which of the following is **not** a remedy for class imbalance when you train a model?

A. Sampling the classes in proportion to their share in the data &nbsp; B. Undersampling the majority class &nbsp; C. Oversampling the minority class &nbsp; D. Re-weighting the loss function

<details><summary><b>Answer</b> (click to unfold)</summary>

**A.** Sampling in proportion to the class shares keeps the imbalance as it is. B, C and D (class weights) are the remedies of Lecture 10.

</details>

**4.** You lower the threshold of a classifier from 0.5 to 0.2. What happens?

A. Recall goes down, precision up &nbsp; B. Recall goes up, precision usually down &nbsp; C. Both go up &nbsp; D. The model has to be trained again

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** More payments are flagged: more of the frauds are found, and more false alarms come in. The model stays the same; only the line moves (Lecture 9, "The threshold is a choice").

</details>

**5.** Which of the following statements about imbalanced data are true?

(a) Undersampling the majority class makes training faster.<br>
(b) Oversampling the majority class helps reduce the complexity of the model.

<details><summary><b>Answer</b> (click to unfold)</summary>

(a) **True:** fewer rows to train on. (b) **False:** you oversample the *minority* class, and that does not make the model simpler (Lecture 10).

</details>

**6.** A colleague applies SMOTE to all 30,000 payments first, and then cross-validates a model on the result. Their recall is

A. honest &nbsp; B. too optimistic &nbsp; C. too pessimistic

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** The synthetic frauds are made from neighbors, some of which end up in the validation folds: the model is tested on near-copies of what it trained on. Resample the training folds only, with the pipeline of imbalanced-learn (Lecture 10, "Resample the training data only").

</details>

**7.** A data scientist builds a classification model on a large, imbalanced data set. The deadline is close, so her main concern is training time. Which remedy for class imbalance should she avoid?

A. Random undersampling &nbsp; B. Random oversampling &nbsp; C. Re-weighting the loss function &nbsp; D. Edited nearest neighbors

<details><summary><b>Answer</b> (click to unfold)</summary>

**B.** Oversampling makes the training data even larger: training is slower, and the copies add no information (Lecture 10, "Random undersampling and oversampling").

</details>

**8.** Which of the following is **not** a metric for regression?

A. Accuracy &nbsp; B. Coefficient of determination ($R^2$) &nbsp; C. Mean squared error &nbsp; D. Mean absolute error

<details><summary><b>Answer</b> (click to unfold)</summary>

**A.** Accuracy counts correct class labels; it is a metric for classification (Lecture 9, "Metrics for regression").

</details>

**9.** You want to compare two models on a rare class before you choose a threshold. Which number do you use?

A. Accuracy &nbsp; B. Recall at 0.5 &nbsp; C. Average precision &nbsp; D. Precision at 0.5

<details><summary><b>Answer</b> (click to unfold)</summary>

**C.** Average precision summarizes the precision-recall curve over all thresholds, so it does not depend on one choice of threshold, and it is not fooled by the many easy negatives (Lecture 9, "Comparing models without choosing a threshold").

</details>

**10.** Below are the true labels `t` and the labels predicted by three classification models `c1`, `c2` and `c3`. Compute the accuracy, and the precision, recall and F1-score for class 1, for each model. You design a theft detection system for a supermarket, and you want to avoid false alarms as much as possible, so that customers are not disturbed for nothing. With 1 meaning theft, which model do you choose, and why?

```
t  = [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
c1 = [0, 0, 0, 1, 0, 0, 0, 1, 1, 0]
c2 = [1, 1, 0, 0, 1, 0, 1, 0, 1, 1]
c3 = [1, 1, 0, 1, 1, 1, 0, 1, 1, 1]
```

<details><summary><b>Answer</b> (click to unfold)</summary>

| model | TP | FP | FN | TN | accuracy | precision | recall | F1 |
|---|---|---|---|---|---|---|---|---|
| c1 | 2 | 1 | 3 | 4 | 0.60 | 0.67 | 0.40 | 0.50 |
| c2 | 3 | 3 | 2 | 2 | 0.50 | 0.50 | 0.60 | 0.55 |
| c3 | 4 | 4 | 1 | 1 | 0.50 | 0.50 | 0.80 | 0.62 |

A false alarm is the costly mistake here, so precision matters most: choose **c1**. When it flags someone, it is right two times out of three. It misses more thefts, but that is the price of disturbing fewer customers (Lecture 9, "Which metric?").

</details>

## If you finish early

**A. The leak, by hand.** Oversample all 30,000 payments with SMOTE first (`SMOTE(random_state=0).fit_resample(X, y)`), then cross-validate a random forest on the result and look at its recall. Then do it the right way, with SMOTE inside `make_imb_pipeline`. Compare (Lecture 10, "Resample the training data only"). This takes about two minutes.

In [ ]:
...

**B. A threshold from costs.** Suppose a check costs the bank €5 of work, and a missed fraud €100. For the logistic regression of Step 1, compute the total cost of every threshold on the honest probabilities `proba`, and find the cheapest. How does its recall compare with the goal of the bank?

In [ ]:
...

## Where this is going

> Choose the metric from what the mistakes cost, compare models with a curve, and choose the threshold last.

Today a rare class made accuracy useless. The threshold turned out to be the strongest lever: class weights and resampling moved the point along the curve, and a better model, not a remedy, raised the part of the curve that the bank uses.

**Coming up.** Week 6 (November 30 to December 3): clustering. For the first time there is no label `y` at all.

**Reading.** Müller and Guido (2016), Chapter 5: "Evaluation Metrics and Scoring". Resampling: imbalanced-learn User Guide, "Over-sampling" and "Under-sampling".

---
*This lab practices Lectures 9 and 10 of Ş. İlker Birbil. Data: "Credit Card Fraud Detection" by the Machine Learning Group of the Université Libre de Bruxelles (Dal Pozzolo, Caelen, Johnson and Bontempi, 2015), from [Kaggle](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud), licensed under the [Open Database License](https://opendatacommons.org/licenses/odbl/1-0/). We use all 492 frauds and a random sample of 49,508 honest payments.*